In [ ]:
import pandas as pd
import numpy as np
import time
import os
n_rows = 2000000 

df = pd.DataFrame({
    'id': range(n_rows),
    'timestamp': pd.date_range('2023-01-01', periods=n_rows, freq='1s'),
    'category': np.random.choice(['A', 'B', 'C', 'D', 'E'], n_rows),
    'value1': np.random.randn(n_rows),
    'value2': np.random.randn(n_rows),
    'value3': np.random.uniform(0, 100, n_rows),
    'text': [f'Transaction_{i}' for i in range(n_rows)]
})

formats = ['csv', 'json', 'parquet', 'feather', 'pickle']
results = {}

for fmt in formats:
    print(f"\nFormatting is in progress: {fmt.upper()}")
    
    start_write = time.time()
    if fmt == 'csv':
        df.to_csv(f'data.{fmt}', index=False)
    elif fmt == 'json':
        df.to_json(f'data.{fmt}', orient='records', lines=True)
    elif fmt == 'parquet':
        df.to_parquet(f'data.{fmt}', index=False)
    elif fmt == 'feather':
        df.to_feather(f'data.{fmt}')
    elif fmt == 'pickle':
        df.to_pickle(f'data.{fmt}')
    write_time = time.time() - start_write

    file_size_mb = os.path.getsize(f'data.{fmt}') / (1024**2)

    start_read = time.time()
    if fmt == 'csv':
        temp_df = pd.read_csv(f'data.{fmt}')
    elif fmt == 'json':
        temp_df = pd.read_json(f'data.{fmt}', orient='records', lines=True)
    elif fmt == 'parquet':
        temp_df = pd.read_parquet(f'data.{fmt}')
    elif fmt == 'feather':
        temp_df = pd.read_feather(f'data.{fmt}')
    elif fmt == 'pickle':
        temp_df = pd.read_pickle(f'data.{fmt}')
    read_time = time.time() - start_read

    start_q = time.time()
    if fmt == 'csv':
        res = temp_df[temp_df['category'] == 'A']['value1'].mean()
    elif fmt == 'json':
        res = temp_df[temp_df['category'] == 'A']['value1'].mean()
    elif fmt == 'parquet':
        q_df = pd.read_parquet(f'data.{fmt}', columns=['category', 'value1'])
        res = q_df[q_df['category'] == 'A']['value1'].mean()
    elif fmt == 'feather':
        q_df = pd.read_feather(f'data.{fmt}', columns=['category', 'value1'])
        res = q_df[q_df['category'] == 'A']['value1'].mean()
    elif fmt == 'pickle':
        res = temp_df[temp_df['category'] == 'A']['value1'].mean()
    query_time = time.time() - start_q

    results[fmt] = {
        'Write Time (s)': write_time,
        'Read Time (s)': read_time,
        'Query Time (s)': query_time,
        'File Size (MB)': file_size_mb
    }

results_df = pd.DataFrame(results).T

results_df['Compression Ratio'] = results_df.loc['csv', 'File Size (MB)'] / results_df['File Size (MB)']


print("Final schedule ")
print(results_df.round(4))